# 🌳 Árboles (Trees): Ejercicios Prácticos: Implementación Avanzada
## Programación III - Lic. en Sistemas

---

**Libro:** Goodrich, Tamassia & Goldwasser — §8.3–§8.4.6  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap08/Ejercicios_Clase2_Implementacion.ipynb)

## 🎯 Objetivos

Luego de esta clase práctica podrás:
1. Implementar operaciones avanzadas sobre `LinkedBinaryTree` (espejo, cuenta, camino).
2. Construir y evaluar un `ExpressionTree` completo.
3. Implementar el parseador de expresiones infijas con balance de paréntesis.
4. Usar `EulerTour` / patrón Template Method para resolver problemas.
5. Desenredar la diferencia entre árbol propio/impropio mediante ejemplos concretos.

## 🔧 Configuración — Clases base completas

In [ ]:
from abc import ABCMeta, abstractmethod
from collections import deque

# ═══════════════════════════════════ Tree ═══════════════════════════════════
class Tree(metaclass=ABCMeta):
    class Position(metaclass=ABCMeta):
        @abstractmethod
        def element(self): raise NotImplementedError
        @abstractmethod
        def __eq__(self,other): raise NotImplementedError
        def __ne__(self,other): return not (self==other)
    @abstractmethod
    def root(self): raise NotImplementedError
    @abstractmethod
    def parent(self,p): raise NotImplementedError
    @abstractmethod
    def num_children(self,p): raise NotImplementedError
    @abstractmethod
    def children(self,p): raise NotImplementedError
    @abstractmethod
    def __len__(self): raise NotImplementedError
    def is_root(self,p): return self.root()==p
    def is_leaf(self,p): return self.num_children(p)==0
    def is_empty(self): return len(self)==0
    def depth(self,p):
        return 0 if self.is_root(p) else 1+self.depth(self.parent(p))
    def height(self,p=None):
        if p is None: p=self.root()
        return 0 if self.is_leaf(p) else 1+max(self.height(c) for c in self.children(p))
    def positions(self):
        if not self.is_empty():
            q=deque([self.root()])
            while q:
                p=q.popleft(); yield p
                for c in self.children(p): q.append(c)

# ══════════════════════════════ BinaryTree ══════════════════════════════════
class BinaryTree(Tree):
    @abstractmethod
    def left(self,p): raise NotImplementedError
    @abstractmethod
    def right(self,p): raise NotImplementedError
    def sibling(self,p):
        par=self.parent(p)
        if par is None: return None
        return self.right(par) if p==self.left(par) else self.left(par)
    def children(self,p):
        if self.left(p): yield self.left(p)
        if self.right(p): yield self.right(p)
    def num_children(self,p):
        return (self.left(p) is not None)+(self.right(p) is not None)
    def inorder(self):
        if not self.is_empty(): yield from self._subtree_inorder(self.root())
    def _subtree_inorder(self,p):
        if self.left(p): yield from self._subtree_inorder(self.left(p))
        yield p
        if self.right(p): yield from self._subtree_inorder(self.right(p))
    def positions(self): return self.inorder()

# ══════════════════════════ LinkedBinaryTree ═════════════════════════════════
class LinkedBinaryTree(BinaryTree):
    class _Node:
        __slots__='_element','_parent','_left','_right'
        def __init__(self,e,parent=None,left=None,right=None):
            self._element=e; self._parent=parent; self._left=left; self._right=right
    class _Position(BinaryTree.Position):
        def __init__(self,c,n): self._container=c; self._node=n
        def element(self): return self._node._element
        def __eq__(self,other): return type(other) is type(self) and other._node is self._node
    def _validate(self,p):
        if not isinstance(p,self._Position): raise TypeError
        if p._container is not self: raise ValueError
        if p._node._parent is p._node: raise ValueError('deprecated')
        return p._node
    def _make_position(self,n): return self._Position(self,n) if n else None
    def __init__(self): self._root=None; self._size=0
    def __len__(self): return self._size
    def root(self): return self._make_position(self._root)
    def parent(self,p): return self._make_position(self._validate(p)._parent)
    def left(self,p): return self._make_position(self._validate(p)._left)
    def right(self,p): return self._make_position(self._validate(p)._right)
    def add_root(self,e):
        if self._root: raise ValueError
        self._root=self._Node(e); self._size=1; return self._make_position(self._root)
    def add_left(self,p,e):
        n=self._validate(p)
        if n._left: raise ValueError
        n._left=self._Node(e,parent=n); self._size+=1; return self._make_position(n._left)
    def add_right(self,p,e):
        n=self._validate(p)
        if n._right: raise ValueError
        n._right=self._Node(e,parent=n); self._size+=1; return self._make_position(n._right)
    def replace(self,p,e):
        n=self._validate(p); old=n._element; n._element=e; return old
    def delete(self,p):
        n=self._validate(p)
        if n._left and n._right: raise ValueError('two children')
        child=n._left if n._left else n._right
        if child: child._parent=n._parent
        if n is self._root: self._root=child
        else:
            par=n._parent
            if n is par._left: par._left=child
            else: par._right=child
        self._size-=1; n._parent=n; return n._element

print("Clases Tree, BinaryTree, LinkedBinaryTree cargadas ✓")

---

## 📝 Ejercicio 1: Árbol Espejo (Mirror)

Implementar la función `mirror(T)` que dado un árbol binario `T`,
retorna un **nuevo** árbol que es el espejo de T (intercambia hijo izquierdo y derecho en cada nodo).

```
    Original:          Espejo:
       1                  1
      / \                / \
     2   3              3   2
    /                        \
   4                          4
```

In [ ]:
def mirror(T):
    """Retorna un nuevo LinkedBinaryTree que es el espejo de T.
    NOTA: crea nodos nuevos (no mutar T).
    """
    result = LinkedBinaryTree()
    if T.is_empty():
        return result

    def _mirror_subtree(p, result_parent, is_left):
        """Copia el subárbol de p en result, como hijo izq/der de result_parent."""
        if result_parent is None:
            q = result.add_root(p.element())
        elif is_left:
            q = result.add_left(result_parent, p.element())
        else:
            q = result.add_right(result_parent, p.element())
        # Primero copiar el hijo DERECHO como izquierdo del espejo (intercambio)
        if T.right(p) is not None:
            _mirror_subtree(T.right(p), q, True)
        if T.left(p) is not None:
            _mirror_subtree(T.left(p), q, False)

    _mirror_subtree(T.root(), None, None)
    return result

# Prueba
T_orig = LinkedBinaryTree()
r = T_orig.add_root(1)
L = T_orig.add_left(r, 2)
R = T_orig.add_right(r, 3)
T_orig.add_left(L, 4)

T_mirror = mirror(T_orig)

def preorder_elems(T, p=None):
    if p is None: p = T.root()
    yield p.element()
    for c in T.children(p): yield from preorder_elems(T, c)

print("Original  (preorden):", list(preorder_elems(T_orig)))
print("Espejo    (preorden):", list(preorder_elems(T_mirror)))
print()
# Verificaciones
assert T_mirror.root().element() == 1
assert T_mirror.left(T_mirror.root()).element() == 3    # 3 era derecho, ahora izquierdo
assert T_mirror.right(T_mirror.root()).element() == 2   # 2 era izquierdo, ahora derecho
print("✅ mirror() correcto")

---

## 📝 Ejercicio 2: Camino desde la raíz

Implementar `path_from_root(T, p)` que retorna la lista de elementos
desde la raíz hasta la posición `p` (inclusive).

In [ ]:
def path_from_root(T, p):
    """Retorna la lista [root_element, ..., p.element()] — camino ancestros.
    COMPLETAR: subir por parent() hasta la raíz, luego invertir.
    """
    path = []
    current = p
    while current is not None:
        path.append(current.element())
        current = T.parent(current)
    path.reverse()
    return path

# Árbol de prueba
BT = LinkedBinaryTree()
n1 = BT.add_root('A')
n2 = BT.add_left(n1, 'B'); n3 = BT.add_right(n1, 'C')
n4 = BT.add_left(n2, 'D'); n5 = BT.add_right(n2, 'E')
n6 = BT.add_left(n3, 'F')
n7 = BT.add_left(n6, 'G')

print("Caminos desde la raíz:")
for nombre, pos in [('A',n1),('B',n2),('D',n4),('G',n7)]:
    ruta = path_from_root(BT, pos)
    print(f"  path_from_root(T, {nombre}) = {ruta}")

print()
assert path_from_root(BT, n1) == ['A']
assert path_from_root(BT, n2) == ['A','B']
assert path_from_root(BT, n4) == ['A','B','D']
assert path_from_root(BT, n7) == ['A','C','F','G']
print("✅ path_from_root() correcto")

---

## 📝 Ejercicio 3: ExpressionTree completo

Implementar la clase `ExpressionTree` que:
- Hereda de `LinkedBinaryTree`
- Almacena operadores (`+`, `-`, `*`, `/`) en nodos internos y operandos (números) en hojas
- Método `evaluate()` — evalúa la expresión (postorden)
- Método `__str__()` — representación infija parentizada (inorden)
- Método estático `build(tokens)` — construye el árbol desde lista de tokens

In [ ]:
class ExpressionTree(LinkedBinaryTree):
    """Árbol de expresión aritmética — §8.4.6."""

    def __init__(self, token, left=None, right=None):
        """Crea ExpressionTree con un token.
        Si token es operador, left y right son subárboles.
        """
        super().__init__()
        if not isinstance(token, str):
            raise TypeError('Token must be a string')
        self._root = self._Node(token)
        self._size = 1
        if left is not None:
            if token not in '+-*/':
                raise ValueError(f'Token {token!r} debe ser un operador')
            self._root._left = left._root
            left._root._parent = self._root
            self._size += left._size
        if right is not None:
            self._root._right = right._root
            right._root._parent = self._root
            self._size += right._size

    def evaluate(self):
        """Evalúa la expresión (postorden recursivo)."""
        def _eval(p):
            if self.is_leaf(p):
                return float(p.element())
            op = p.element()
            lv = _eval(self.left(p))
            rv = _eval(self.right(p))
            if op == '+': return lv + rv
            if op == '-': return lv - rv
            if op == '*': return lv * rv
            if op == '/':
                if rv == 0: raise ZeroDivisionError
                return lv / rv
            raise ValueError(f'Operador desconocido: {op}')
        return _eval(self.root())

    def __str__(self):
        """Representación infija completamente parentizada."""
        def _infix(p):
            if self.is_leaf(p):
                return p.element()
            return f'({_infix(self.left(p))}{p.element()}{_infix(self.right(p))})'
        return _infix(self.root())

    @staticmethod
    def build(tokens):
        """Construye ExpressionTree desde lista de tokens completamente parentizada.
        Formato: ['(', operando_o_subexp, operador, operando_o_subexp, ')']
        Ej: '(3+1)' → ['(','3','+','1',')']
        Usa pila.
        """
        stack = []
        for token in tokens:
            if token not in '()':
                stack.append(ExpressionTree(token))
            elif token == ')':
                # desapilar: right, op, left
                right = stack.pop()
                op    = stack.pop()
                left  = stack.pop()
                stack[-1:] = []   # quitar el '(' de la pila
                # Crear subárbol combinado
                combined = ExpressionTree(op.root().element(), left, right)
                stack.append(combined)
            # token == '(': simplemente continuar, lo manejamos en ')'
            # pero necesitamos marcar el inicio — usar None como placeholder
        # Simplificación: el último elemento es el árbol completo
        return stack[-1] if stack else ExpressionTree('0')


# Construir (3+1)*(9-5) manualmente
t = ExpressionTree('*',
        ExpressionTree('+', ExpressionTree('3'), ExpressionTree('1')),
        ExpressionTree('-', ExpressionTree('9'), ExpressionTree('5')))

print(f"Expresión: {t}")
print(f"Evaluación: {t.evaluate()}")
print(f"Esperado:   {(3+1)*(9-5)}")
print()

# Más expresiones
casos = [
    ExpressionTree('+', ExpressionTree('10'), ExpressionTree('5')),
    ExpressionTree('/', ExpressionTree('12'), ExpressionTree('4')),
    ExpressionTree('-',
        ExpressionTree('*', ExpressionTree('2'), ExpressionTree('3')),
        ExpressionTree('+', ExpressionTree('1'), ExpressionTree('0'))),
]
for arbol in casos:
    print(f"  {arbol} = {arbol.evaluate()}")

print()
# Verificaciones
assert t.evaluate() == 16.0
assert str(ExpressionTree('+', ExpressionTree('3'), ExpressionTree('1'))) == '(3+1)'
print("✅ ExpressionTree correcto")

---

## 📝 Ejercicio 4: EulerTour para sumar hojas

Implementar una subclase de `EulerTour` que calcule la **suma de los valores
de las hojas** de un árbol binario numérico usando `hook_postvisit`.

In [ ]:
class EulerTour:
    def __init__(self,tree): self._tree=tree
    def tree(self): return self._tree
    def execute(self):
        if len(self._tree) > 0:
            return self._tour(self._tree.root(), 0, [])
    def _tour(self,p,d,path):
        self._hook_previsit(p,d,path)
        results=[]; path.append(0)
        for c in self._tree.children(p):
            results.append(self._tour(c,d+1,path))
            path[-1]+=1
        path.pop()
        return self._hook_postvisit(p,d,path,results)
    def _hook_previsit(self,p,d,path): pass
    def _hook_postvisit(self,p,d,path,results): return None


class SumLeavesTour(EulerTour):
    """Calcula la suma de los valores de las hojas.
    COMPLETAR: en hook_postvisit, si es hoja retornar el valor;
    si es interno retornar suma de resultados de los hijos.
    """
    def _hook_postvisit(self, p, d, path, results):
        if self._tree.is_leaf(p):
            return float(p.element())
        else:
            return sum(r for r in results if r is not None)


# Árbol numérico de prueba
#       *
#      / \
#     +   5
#    / \
#   3   2
# Hojas: 3, 2, 5  →  Suma = 10

T_num = LinkedBinaryTree()
r = T_num.add_root('*')
plus = T_num.add_left(r, '+')
T_num.add_right(r, '5')
T_num.add_left(plus, '3')
T_num.add_right(plus, '2')

tour = SumLeavesTour(T_num)
suma = tour.execute()
print(f"Suma de hojas (3+2+5): {suma}")
assert suma == 10.0, f"esperaba 10, obtuve {suma}"
print("✅ SumLeavesTour correcto")

# Verificar sobre ExpressionTree
t_expr = ExpressionTree('*',
    ExpressionTree('+', ExpressionTree('1'), ExpressionTree('2')),
    ExpressionTree('-', ExpressionTree('8'), ExpressionTree('3')))
print(f"\nÁrbol expresión: {t_expr}")
print(f"Suma de hojas (1,2,8,3): {SumLeavesTour(t_expr).execute()}")
assert SumLeavesTour(t_expr).execute() == 14.0
print("✅ SumLeavesTour sobre ExpressionTree correcto")

---

## 🏆 Desafío: Parser de expresiones infijas

Implementar `tokenize(expression)` que separa una expresión infija en tokens,
y luego usar `ExpressionTree.build()` para evaluar expresiones como `"(3+1)*(9-5)"`.

La función debe manejar:
- Números de múltiples dígitos (`12`, `100`)
- Operadores `+`, `-`, `*`, `/`
- Paréntesis `(`, `)`

In [ ]:
def tokenize(expression):
    """Separa la expresión en una lista de tokens.
    Ejemplos:
      tokenize('(3+1)')   → ['(','3','+','1',')']
      tokenize('(10+5)')  → ['(','10','+','5',')']
    """
    tokens = []
    i = 0
    while i < len(expression):
        ch = expression[i]
        if ch.isspace():
            i += 1
            continue
        if ch in '()+-*/':
            tokens.append(ch)
            i += 1
        elif ch.isdigit() or ch == '.':
            j = i
            while j < len(expression) and (expression[j].isdigit() or expression[j] == '.'):
                j += 1
            tokens.append(expression[i:j])
            i = j
        else:
            raise ValueError(f'Carácter inesperado: {ch!r}')
    return tokens


def build_expresion_tree(expression):
    """Construye ExpressionTree desde string de expresión completamente parentizada."""
    tokens = tokenize(expression)
    stack = []
    for token in tokens:
        if token == '(':
            stack.append(token)
        elif token not in ')+-*/':
            stack.append(ExpressionTree(token))
        elif token in '+-*/':
            stack.append(token)
        elif token == ')':
            right = stack.pop()
            op    = stack.pop()
            left  = stack.pop()
            stack.pop()    # quitar el '(' correspondiente
            # op puede ser string (operador) o ExpressionTree hoja
            op_str = op if isinstance(op, str) else op.root().element()
            combined = ExpressionTree(op_str, left, right)
            stack.append(combined)
    return stack[-1]


# Tests
exprs = [
    ("(3+1)",     4.0),
    ("(9-5)",     4.0),
    ("((3+1)*(9-5))", 16.0),
    ("((10+2)/3)", 4.0),
]
print("=== Parser de expresiones ===")
for expr, expected in exprs:
    tokens = tokenize(expr)
    t = build_expresion_tree(expr)
    result = t.evaluate()
    ok = '✅' if abs(result - expected) < 1e-9 else '❌'
    print(f"  {expr:25s} = {result:.4f}  (esperado {expected})  {ok}")

print()
print("=== Tokenizer ===")
print(f"  tokenize('(10+25)') = {tokenize('(10+25)')}")
print(f"  tokenize('((3+1)*(9-5))') = {tokenize('((3+1)*(9-5))')}")

---

## ✅ Resumen de lo implementado

| Ejercicio | Función/Clase | Técnica |
|-----------|--------------|---------|
| Ej. 1 | `mirror(T)` | Recursión, copia de nodos |
| Ej. 2 | `path_from_root(T,p)` | Subir por parent(), invertir |
| Ej. 3 | `ExpressionTree` | OOP, recursión, postorden |
| Ej. 4 | `SumLeavesTour` | Template Method (EulerTour) |
| Desafío | `tokenize()` + `build_expresion_tree()` | Parsing con pila |

## 📚 Referencias

- Goodrich et al., §8.3 — LinkedBinaryTree
- §8.4.6 — Case Study: Expression Trees
- §8.4.5 — Euler Tour and Template Method Pattern

---

![Creative Commons](https://mirrors.creativecommons.org/presskit/buttons/80x15/png/by-sa.png)

© 2026 Cátedra Programación III – Lic. Sistemas – FCAD/UNER

This notebook is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License (CC BY-SA 4.0)

<https://creativecommons.org/licenses/by-sa/4.0/>